# Notebook 07 - Vina Cross-Docking

**Inputs (from notebooks 02 + 03 + 05):**
- `data/pdb/1M17_receptor.pdbqt`, `data/pdb/4I22_receptor.pdbqt`  (prepared in 03)
- `data/pocket_centers.json`                                       (computed in 02)
- `results/filtered/1M17_filtered.csv`  (264 WT-generated drug-like mols)
- `results/filtered/4I22_filtered.csv`  (284 mutant-generated drug-like mols)
- `results/filtered/baselines.csv`      (Erlotinib / Gefitinib / Osimertinib)

**What this notebook does (the academic-finding step):**

Each filtered molecule is docked into BOTH the WT and the T790M pocket, producing
a `(vina_WT, vina_Mut)` score pair. This 2-D number per molecule is the input to
the WT-vs-Mutant **selectivity scatter** in notebook 09 - the central figure of
the project.

**Output (in `results/vina_scores/`):**
- `wt_pool_cross.csv`    - WT-generated pool docked against both pockets
- `mut_pool_cross.csv`   - Mutant-generated pool docked against both pockets
- `baseline_cross.csv`   - 3 approved drugs docked against both pockets
- `combined_scores.csv`  - everything merged with a `source` column

**Runtime:** CPU only. ~3.5 hours at exhaustiveness=4 on Colab CPU, top 50 mols per pool.
The notebook is **resumable**: each docking is checkpointed every 25 molecules,
so if the session disconnects, re-running picks up where it left off.

**Why exhaustiveness=4 + Top-50:** Colab CPU benchmark gave ~117s/dock at
exh=8. Trimming to top-50 by QED per pool + halving exhaustiveness brings
the full run to ~3.5h. Notebook 08 will re-dock only the Top-20 hits at
exhaustiveness=32 for higher-confidence final scores.


In [ ]:
# -- Cell 1: Bootstrap (CPU runtime; install Vina + obabel + rdkit) -------------
import os, sys, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
except ImportError:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

# Pull latest code
subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=False)

# Vina + Open Babel + rdkit (idempotent: skip if already installed)
def _have(cmd): return subprocess.run(['which', cmd], capture_output=True, text=True).stdout.strip()
if not _have('vina'):
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'autodock-vina'], check=False)
if not _have('obabel'):
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openbabel'], check=False)
try:
    import rdkit
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'tqdm'], check=True)
    import rdkit

print('vina   :', _have('vina')   or 'MISSING')
print('obabel :', _have('obabel') or 'MISSING')
print('rdkit  :', rdkit.__version__)


## Step 1 - Verify required artifacts are on Drive

Receptor PDBQTs and pocket centers come from earlier notebooks. If anything is
missing, stop here and re-run notebook 02 / 03.


In [ ]:
# -- Cell 2: Verify receptor PDBQT + pocket centers exist --
import os, json

required = {
    '1M17 receptor PDBQT': f'{PROJECT_ROOT}/data/pdb/1M17_receptor.pdbqt',
    '4I22 receptor PDBQT': f'{PROJECT_ROOT}/data/pdb/4I22_receptor.pdbqt',
    'pocket centers JSON': f'{PROJECT_ROOT}/data/pocket_centers.json',
    'WT  filtered CSV'    : f'{PROJECT_ROOT}/results/filtered/1M17_filtered.csv',
    'Mut filtered CSV'    : f'{PROJECT_ROOT}/results/filtered/4I22_filtered.csv',
    'Baselines CSV'       : f'{PROJECT_ROOT}/results/filtered/baselines.csv',
}
missing = []
for label, path in required.items():
    ok = os.path.exists(path)
    sz = os.path.getsize(path)/1024 if ok else 0
    print(f'  {label:25s} {"✓" if ok else "✗"}  {path}  ({sz:.1f} KB)')
    if not ok: missing.append(label)
assert not missing, f'Missing artifacts: {missing}. Re-run earlier notebooks first.'

with open(required['pocket centers JSON']) as f:
    pocket_centers = json.load(f)
print('\nPocket centers:', pocket_centers)


## Step 2 - Build the two `VinaDocker` instances

In [ ]:
# -- Cell 3: Build dockers for WT (1M17) and Mutant (4I22) pockets --
from src.docking import VinaDocker

EXHAUSTIVENESS = 8       # matches TargetDiff paper screening setting
BOX_SIZE       = (20.0, 20.0, 20.0)
TOP_K_BY_QED   = 50      # take top 50 by QED from each filtered pool (~6.7 h on Colab CPU)

docker_wt = VinaDocker(
    receptor_pdbqt=f'{PROJECT_ROOT}/data/pdb/1M17_receptor.pdbqt',
    center=tuple(pocket_centers['1M17']),
    box_size=BOX_SIZE,
    exhaustiveness=EXHAUSTIVENESS,
)
docker_mut = VinaDocker(
    receptor_pdbqt=f'{PROJECT_ROOT}/data/pdb/4I22_receptor.pdbqt',
    center=tuple(pocket_centers['4I22']),
    box_size=BOX_SIZE,
    exhaustiveness=EXHAUSTIVENESS,
)
print('WT  docker: receptor=', docker_wt.receptor,  '| center=', docker_wt.center)
print('Mut docker: receptor=', docker_mut.receptor, '| center=', docker_mut.center)
print()
print(f'Config: top-{TOP_K_BY_QED} per pool by QED, exhaustiveness={EXHAUSTIVENESS}')
print(f'Expected: 50 + 50 + 3 = 103 mols x 2 pockets = 206 dockings')
print(f'Estimated runtime on Colab CPU: ~6.7 hours')


## Step 3 - Smoke test: dock Erlotinib against both pockets

This is a 30-second sanity check before committing to the full ~3-hour run.
Erlotinib is a known WT-selective drug, so we expect:
- `vina_WT  ~ -8 to -11`  (strong)
- `vina_Mut ~ -6 to -9`   (weaker than WT, since T790M is the mutation that *resists* Erlotinib)


In [ ]:
# -- Cell 4: Smoke test on Erlotinib --
from rdkit import Chem
import time

erlotinib_smi = 'C(#N)c1cc2c(cc1OCC)NC(=O)c1cc(OCC)c(OCC)c(OC)c1N2'  # actually that's not erlotinib smile
# Use the canonical SMILES from the project config
import yaml
with open(f'{PROJECT_ROOT}/configs/targets.yaml') as f:
    cfg = yaml.safe_load(f)
erlotinib_smi = cfg['baselines']['erlotinib']['smiles']
mol = Chem.MolFromSmiles(erlotinib_smi)
print('Erlotinib SMILES:', erlotinib_smi)

t0 = time.time()
score_wt  = docker_wt.dock_mol(mol)
score_mut = docker_mut.dock_mol(mol)
print(f'\nErlotinib vina_WT  = {score_wt}  kcal/mol')
print(f'Erlotinib vina_Mut = {score_mut}  kcal/mol')
print(f'Elapsed: {time.time()-t0:.1f}s for 2 dockings')

if score_wt is None or score_mut is None:
    raise RuntimeError('Smoke test failed - check Vina install + receptor preparation.')
print('\n✓ Smoke test passed. Proceed to full cross-docking.')


## Step 4 - Cross-dock top-50 WT-generated mols (~85 min)

`cross_dock_smiles_list` is **resumable**: it checkpoints every 25 molecules.
If Colab disconnects, just re-run this cell - already-done mols are skipped.


In [ ]:
# -- Cell 5: Cross-dock top-50 WT-generated mols by QED --
import pandas as pd
from src.docking import cross_dock_smiles_list

wt_filt = pd.read_csv(f'{PROJECT_ROOT}/results/filtered/1M17_filtered.csv')
print(f'WT pool (filtered): {len(wt_filt)} mols')
wt_filt = wt_filt.nlargest(TOP_K_BY_QED, 'QED').reset_index(drop=True)
print(f'-> taking top {TOP_K_BY_QED} by QED for cross-docking: {len(wt_filt)} mols')

wt_results = cross_dock_smiles_list(
    smiles_list=wt_filt['smiles'].tolist(),
    docker_wt=docker_wt,
    docker_mut=docker_mut,
    output_csv=f'{PROJECT_ROOT}/results/vina_scores/wt_pool_cross.csv',
    source_label='WT_generated',
    checkpoint_every=10,
)
wt_results.head()


## Step 5 - Cross-dock top-50 Mutant-generated mols (~85 min)

Same pattern as Step 4 but on the mutant pool. Resumable.


In [ ]:
# -- Cell 6: Cross-dock top-50 Mutant-generated mols by QED --
mut_filt = pd.read_csv(f'{PROJECT_ROOT}/results/filtered/4I22_filtered.csv')
print(f'Mutant pool (filtered): {len(mut_filt)} mols')
mut_filt = mut_filt.nlargest(TOP_K_BY_QED, 'QED').reset_index(drop=True)
print(f'-> taking top {TOP_K_BY_QED} by QED for cross-docking: {len(mut_filt)} mols')

mut_results = cross_dock_smiles_list(
    smiles_list=mut_filt['smiles'].tolist(),
    docker_wt=docker_wt,
    docker_mut=docker_mut,
    output_csv=f'{PROJECT_ROOT}/results/vina_scores/mut_pool_cross.csv',
    source_label='Mut_generated',
    checkpoint_every=10,
)
mut_results.head()


## Step 6 - Cross-dock the 3 approved drugs (baselines)

Only 3 molecules x 2 pockets = 6 dockings, ~30 seconds. These will become the
reference points on the selectivity scatter (Erlotinib in WT-selective quadrant,
Osimertinib in Mut-selective quadrant, etc.).


In [ ]:
# -- Cell 7: Cross-dock baselines --
baselines = pd.read_csv(f'{PROJECT_ROOT}/results/filtered/baselines.csv')
print(f'Baselines: {len(baselines)} molecules ({list(baselines["name"]) if "name" in baselines else baselines.columns.tolist()})')

baseline_results = cross_dock_smiles_list(
    smiles_list=baselines['smiles'].tolist(),
    docker_wt=docker_wt,
    docker_mut=docker_mut,
    output_csv=f'{PROJECT_ROOT}/results/vina_scores/baseline_cross.csv',
    source_label='baseline',
    checkpoint_every=1,
)

# Attach the drug name back
if 'name' in baselines.columns:
    baseline_results = baseline_results.merge(
        baselines[['smiles', 'name']], on='smiles', how='left'
    )
    out_path = f'{PROJECT_ROOT}/results/vina_scores/baseline_cross.csv'
    baseline_results.to_csv(out_path, index=False)

print(baseline_results[['smiles','vina_WT','vina_Mut'] +
                      (['name'] if 'name' in baseline_results else [])])


## Step 7 - Merge into a single `combined_scores.csv`

Concatenate WT + Mut + baseline rows so notebook 09 can plot everything from one
DataFrame. We also drop molecules that failed docking on either pocket (very rare).


In [ ]:
# -- Cell 8: Build combined_scores.csv --
import pandas as pd
import os

vs_dir = f'{PROJECT_ROOT}/results/vina_scores'
parts = []
for fname, label in [('wt_pool_cross.csv',   'WT_generated'),
                     ('mut_pool_cross.csv',  'Mut_generated'),
                     ('baseline_cross.csv',  'baseline')]:
    p = os.path.join(vs_dir, fname)
    if not os.path.exists(p):
        print(f'WARN: {p} missing, skipping')
        continue
    df = pd.read_csv(p)
    if 'source' not in df:
        df['source'] = label
    parts.append(df)

combined = pd.concat(parts, ignore_index=True)

# Keep only successful dockings on both pockets
ok = (combined['status_WT'] == 'ok') & (combined['status_Mut'] == 'ok')
combined_ok = combined[ok].copy()

print(f'Total rows           : {len(combined)}')
print(f'Successful (both ok) : {len(combined_ok)}  ({100*ok.mean():.1f}%)')
print(f'\nPer-source counts:')
print(combined_ok['source'].value_counts())

out_path = f'{vs_dir}/combined_scores.csv'
combined_ok.to_csv(out_path, index=False)
print(f'\nSaved -> {out_path}')


## Step 8 - Quick scatter preview

A first look at the WT vs Mutant selectivity plot. Notebook 09 will polish this
with annotations, quadrant lines, and Tanimoto-color-coded points; here we just
sanity-check that the data is sensible (e.g., baselines fall on the right side).


In [ ]:
# -- Cell 9: Preview the WT-vs-Mutant scatter --
import matplotlib.pyplot as plt

color_map = {'WT_generated':'#1f77b4', 'Mut_generated':'#d62728', 'baseline':'#2ca02c'}

fig, ax = plt.subplots(figsize=(6.5, 6.5))
for src, color in color_map.items():
    sub = combined_ok[combined_ok['source'] == src]
    ax.scatter(sub['vina_WT'], sub['vina_Mut'],
               c=color, label=f'{src} (n={len(sub)})',
               alpha=0.5 if src != 'baseline' else 1.0,
               s=20 if src != 'baseline' else 120,
               edgecolor='black' if src == 'baseline' else 'none')

# Annotate baselines by name
if 'name' in combined_ok.columns:
    for _, row in combined_ok[combined_ok['source'] == 'baseline'].iterrows():
        if pd.notna(row.get('name')):
            ax.annotate(row['name'], (row['vina_WT'], row['vina_Mut']),
                        xytext=(7, 5), textcoords='offset points', fontsize=9)

# Diagonal: equal binding to both pockets
lo = min(combined_ok['vina_WT'].min(), combined_ok['vina_Mut'].min()) - 0.5
hi = max(combined_ok['vina_WT'].max(), combined_ok['vina_Mut'].max()) + 0.5
ax.plot([lo, hi], [lo, hi], '--', color='gray', alpha=0.5, label='equal binding')

ax.set_xlabel('Vina score on WT (1M17), kcal/mol  (lower = better)')
ax.set_ylabel('Vina score on T790M (4I22), kcal/mol  (lower = better)')
ax.set_title('Cross-docking preview - WT vs Mutant binding')
ax.invert_xaxis(); ax.invert_yaxis()  # flip so 'better' is upper-right
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## What's next

- **Notebook 08** (ranking + Top-20 cards): use `combined_scores.csv` to score every molecule
  on QED + SA + vina + novelty, pick top 20, generate per-molecule report cards.
- **Notebook 09** (selectivity scatter): polish the figure above with quadrant labels,
  Tanimoto-to-baseline colormap, and a discussion of what the model has implicitly learned.
